# 🛡️ VIGILANT — Viral Infant Guardian: Intelligent Linkage and Adherence Network for Treatment

**Gemma 4 Good Hackathon | Health & Sciences Track + Ollama Special Technology Track**

VIGILANT uses **Gemma 4 via Ollama** to protect HIV-exposed newborns in Sub-Saharan Africa by:
1. **Forensic Linker** — Matching infants to mothers across fragmented records
2. **Adherence Miner** — Extracting medication adherence risks from clinical notes using Gemma 4 native function calling
3. **Protocol Guardian** — Classifying infant risk and recommending prophylaxis regimens

All processing runs **locally** via Ollama — no patient data leaves the device (NDPA 2023 compliant).

## 1. Setup & Dependencies

In [ ]:
import os, subprocess, time, requests
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

MODEL_NAME = 'gemma4:e4b'
OLLAMA_BASE_URL = 'http://localhost:11434'

!pip install -q rapidfuzz ollama

# Install dependencies and Ollama (Linux/Kaggle)
!apt-get install -q -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background with GPU support
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until Ollama is actually ready (up to 30s)
for i in range(30):
    try:
        if requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=2).status_code == 200:
            print(f'✅ Ollama ready after {i+1}s')
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start within 30 seconds')

# Pull the actual Gemma 4 edge model tag published by Ollama
subprocess.run(['ollama', 'pull', MODEL_NAME], check=True)

# Warm up the model once so the first real inference does not pay full load cost
warmup_payload = {
    'model': MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Reply with OK'}],
    'stream': False,
    'keep_alive': '30m',
    'options': {'temperature': 0, 'num_predict': 8},
}
requests.post(f'{OLLAMA_BASE_URL}/api/chat', json=warmup_payload, timeout=(5, 180)).raise_for_status()

# Confirm model is listed
models = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=10).json().get('models', [])
model_names = [m['name'] for m in models]
print('Models available:', model_names)
if MODEL_NAME not in model_names:
    raise RuntimeError(f'{MODEL_NAME} was not found after pull')
print(f'✅ Ollama + {MODEL_NAME} ready')


## 2. Data Schemas

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Evidence:
    type: str
    detail: str
    strength: float

@dataclass
class LinkageResult:
    mother_id: str
    mother_name: str
    art_id: str
    confidence: float
    evidence: list

@dataclass
class AdherenceRisk:
    indicator: str
    source_text: str
    source_date: str
    severity: str = "moderate"

@dataclass
class RiskClassification:
    level: str
    reasons: list
    viral_load: float
    viral_load_date: str
    adherence_risks: list

@dataclass
class BridgeSummary:
    infant_name: str
    mother_name: str
    art_id: str
    confidence: float
    evidence_summary: str
    viral_load: float
    risk_level: str
    adherence_findings: list
    recommended_action: str
    audit_hash: str = ''

print('✅ Schemas defined')

## 3. Synthetic Data Generator
Generates 50 FHIR-like mother/infant records with realistic African names, dirty data, and varied risk profiles.

In [ ]:
import json, uuid, random, copy
from datetime import datetime, timedelta

random.seed(42)

FIRST_NAMES = ["Mary","Grace","Esther","Ruth","Mercy","Faith","Joy","Blessing","Patience","Agnes",
               "Florence","Rose","Sarah","Amina","Fatima","Aisha","Ngozi","Chiamaka","Wanjiku","Nia"]
LAST_NAMES = ["Banda","Moyo","Phiri","Mwale","Tembo","Nkosi","Okafor","Adeyemi","Muthoni","Ochieng",
              "Dlamini","Ndlovu","Kamau","Mensah","Asante","Diallo","Traore","Keita","Balogun","Eze"]
FACILITIES = ["Kenyatta National Hospital","Queen Elizabeth Central Hospital","Muhimbili National Hospital",
              "Lagos University Teaching Hospital","Korle Bu Teaching Hospital"]

def misspell(name):
    if len(name) < 3: return name
    op = random.choice(["swap","drop","double"])
    i = random.randint(1, len(name)-2)
    if op == "swap":
        lst = list(name); lst[i], lst[i-1] = lst[i-1], lst[i]; return "".join(lst)
    elif op == "drop": return name[:i] + name[i+1:]
    else: return name[:i] + name[i] + name[i:]

NORMAL_NOTES = [
    "Patient attended ANC visit. All vitals normal. ART adherence confirmed.",
    "Routine follow-up. Patient reports no issues. Pharmacy refill collected on time.",
    "Antenatal visit. Fetal growth normal. Patient compliant with medication.",
]
RISKY_NOTES = [
    "Patient missed pharmacy pick-up on {date}. Reports transport difficulties.",
    "Patient did not attend scheduled ANC visit. Called — reports financial constraints.",
    "Late presentation to antenatal care (first visit at 32 weeks). Adherence counseling provided.",
    "Patient reports missing ART doses for 5 days due to travel. Counseled on adherence.",
    "Pharmacy records show 2 missed pick-ups in the last 3 months. Patient cites transport challenges.",
    "Patient admitted to skipping medication when feeling well. Re-educated on importance of adherence.",
]

def generate_phone(): return f"+234-{random.randint(700,999)}-{random.randint(1000,9999)}"

def generate_mothers(n=50):
    mothers = []
    for i in range(n):
        first, last = random.choice(FIRST_NAMES), random.choice(LAST_NAMES)
        dob = datetime(1985+random.randint(0,15), random.randint(1,12), random.randint(1,28))
        facility, phone = random.choice(FACILITIES), generate_phone()
        art_id = f"ART-{random.randint(1000,9999)}"
        delivery_date = datetime(2026, 3, random.randint(1,31)) if random.random() > 0.3 else datetime(2026, 4, random.randint(1,7))
        if i < 20:
            vl_value = random.randint(1500, 80000)
            vl_date = (delivery_date - timedelta(days=random.randint(7,60))).isoformat()
            notes = [random.choice(NORMAL_NOTES)]; risk_category = "OBVIOUS_HIGH"
        elif i < 30:
            vl_value = random.randint(20, 180)
            vl_date = (delivery_date - timedelta(days=random.randint(7,60))).isoformat()
            note_date = (delivery_date - timedelta(days=random.randint(5,30))).strftime("%Y-%m-%d")
            notes = [t.format(date=note_date) if "{date}" in t else t for t in random.sample(RISKY_NOTES, k=random.randint(2,3))]
            risk_category = "HIDDEN_HIGH"
        elif i < 35:
            vl_value = None; vl_date = ""; notes = [random.choice(NORMAL_NOTES)]; risk_category = "MISSING_VL"
        elif i < 40:
            vl_value = random.randint(50, 1000)
            vl_date = (delivery_date - timedelta(days=random.randint(7,60))).isoformat()
            notes = [random.choice(NORMAL_NOTES)]; risk_category = "MODERATE"
        else:
            vl_value = random.randint(0, 40)
            vl_date = (delivery_date - timedelta(days=random.randint(7,30))).isoformat()
            notes = [random.choice(NORMAL_NOTES)]; risk_category = "LOW"
        mother = {
            "resourceType": "Patient", "id": str(uuid.uuid4()),
            "name": [{"family": last, "given": [first]}], "gender": "female",
            "birthDate": dob.strftime("%Y-%m-%d"),
            "telecom": [{"system": "phone", "value": phone}],
            "identifier": [{"system": "urn:art-program", "value": art_id}],
            "meta": {"facility": facility, "delivery_date": delivery_date.isoformat(), "risk_category": risk_category},
            "viral_load": {
                "resourceType": "Observation",
                "code": {"coding": [{"system": "http://loinc.org", "code": "20447-9", "display": "HIV 1 RNA"}]},
                "valueQuantity": {"value": vl_value, "unit": "copies/mL"} if vl_value is not None else {},
                "effectiveDateTime": vl_date
            },
            "clinical_notes": [{"resourceType": "DocumentReference", "content": note,
                                 "date": (delivery_date - timedelta(days=random.randint(1,30))).isoformat()} for note in notes],
            "condition": {"resourceType": "Condition",
                          "code": {"coding": [{"system": "http://snomed.info", "code": "86406008", "display": "HIV disease"}]},
                          "clinicalStatus": "active"}
        }
        mothers.append(mother)
    return mothers

def generate_newborns(mothers):
    newborns = []
    for i, mom in enumerate(mothers):
        first_name, last_name = mom["name"][0]["given"][0], mom["name"][0]["family"]
        facility, delivery, phone = mom["meta"]["facility"], mom["meta"]["delivery_date"], mom["telecom"][0]["value"]
        style = random.choice(["baby_of","baby_of_misspell","unnamed","first_name_only"])
        if style == "baby_of": infant_name = {"family": last_name, "given": [f"Baby of {first_name}"]}
        elif style == "baby_of_misspell": infant_name = {"family": misspell(last_name), "given": [f"Baby of {first_name}"]}
        elif style == "unnamed": infant_name = {"family": "", "given": [random.choice(["Unnamed Male","Unnamed Female","Baby Boy","Baby Girl"])]}
        else: infant_name = {"family": last_name, "given": [first_name]}
        if i < 5: phone = "+234-800-SHARED"; mom["telecom"] = [{"system": "phone", "value": phone}]
        infant_phone = phone if random.random() > 0.3 else ""
        newborn = {
            "resourceType": "Patient", "id": str(uuid.uuid4()),
            "name": [infant_name], "gender": random.choice(["male","female"]),
            "birthDate": delivery[:10],
            "telecom": [{"system": "phone", "value": infant_phone}] if infant_phone else [],
            "meta": {"facility": facility if random.random() > 0.1 else random.choice(FACILITIES), "mother_id": mom["id"]}
        }
        newborns.append(newborn)
    return newborns

mothers = generate_mothers(50)
newborns = generate_newborns(mothers)
print(f"Generated {len(mothers)} mothers and {len(newborns)} newborns")
print(f"  Obvious HIGH: {sum(1 for m in mothers if m['meta']['risk_category']=='OBVIOUS_HIGH')}")
print(f"  Hidden HIGH:  {sum(1 for m in mothers if m['meta']['risk_category']=='HIDDEN_HIGH')}")
print(f"  Missing VL:   {sum(1 for m in mothers if m['meta']['risk_category']=='MISSING_VL')}")
print(f"  MODERATE:     {sum(1 for m in mothers if m['meta']['risk_category']=='MODERATE')}")
print(f"  LOW:          {sum(1 for m in mothers if m['meta']['risk_category']=='LOW')}")
print(f"  Shared phone: {sum(1 for n in newborns if any(t.get('value')=='+234-800-SHARED' for t in n.get('telecom',[])))}")

## 4. Agent 1 — Forensic Linker
Matches infants to mothers using fuzzy name matching, facility, birth timing, and phone — with **Shared Phone Penalty**. Returns **Top 3 candidates**.

In [ ]:
from rapidfuzz import fuzz

def extract_mother_name_from_infant(infant):
    given = infant["name"][0].get("given", [""])[0]
    family = infant["name"][0].get("family", "")
    if given.lower().startswith("baby of "):
        return given[8:].strip(), family
    if given.lower() in ["unnamed male","unnamed female","baby boy","baby girl"]:
        return None, family
    return given, family

def _build_phone_usage_map(mothers_list):
    phone_counts = {}
    for mother in mothers_list:
        for t in mother.get("telecom", []):
            phone = t.get("value", "")
            if phone: phone_counts[phone] = phone_counts.get(phone, 0) + 1
    return phone_counts

def score_match(infant, mother, phone_usage_map=None):
    evidence = []; total = 0
    extracted_first, extracted_family = extract_mother_name_from_infant(infant)
    mother_first = mother["name"][0]["given"][0]
    mother_family = mother["name"][0]["family"]
    if extracted_first:
        first_score = fuzz.ratio(extracted_first.lower(), mother_first.lower())
        if first_score > 70:
            evidence.append(Evidence("name_similarity", f"First name: '{extracted_first}'~'{mother_first}' ({first_score}%)", first_score/100))
            total += first_score * 0.3
    if extracted_family:
        family_score = fuzz.ratio(extracted_family.lower(), mother_family.lower())
        if family_score > 70:
            evidence.append(Evidence("name_similarity", f"Family name: '{extracted_family}'~'{mother_family}' ({family_score}%)", family_score/100))
            total += family_score * 0.25
    infant_facility = infant.get("meta",{}).get("facility","")
    mother_facility = mother.get("meta",{}).get("facility","")
    if infant_facility and mother_facility and infant_facility == mother_facility:
        evidence.append(Evidence("facility_match", f"Same facility: {infant_facility}", 1.0)); total += 20
    infant_dob = infant.get("birthDate","")
    mother_delivery = mother.get("meta",{}).get("delivery_date","")[:10]
    if infant_dob and mother_delivery:
        try:
            days_diff = abs((datetime.strptime(infant_dob,"%Y-%m-%d") - datetime.strptime(mother_delivery,"%Y-%m-%d")).days)
            if days_diff <= 1: evidence.append(Evidence("birth_timing", f"Birth within {days_diff} day(s)", 1.0)); total += 20
            elif days_diff <= 3: evidence.append(Evidence("birth_timing", f"Birth within {days_diff} days", 0.7)); total += 10
        except ValueError: pass
    infant_phones = [t["value"] for t in infant.get("telecom",[]) if t.get("value")]
    mother_phones = [t["value"] for t in mother.get("telecom",[]) if t.get("value")]
    if infant_phones and mother_phones:
        for ip in infant_phones:
            for mp in mother_phones:
                if ip == mp:
                    phone_count = phone_usage_map.get(mp, 1) if phone_usage_map else 1
                    if phone_count > 1:
                        pw = max(5, 15 // phone_count)
                        evidence.append(Evidence("phone_match", f"Phone {ip} (SHARED by {phone_count}, weight={pw})", pw/15))
                        total += pw
                    else:
                        evidence.append(Evidence("phone_match", f"Phone match: {ip}", 1.0)); total += 15
                    break
    return min(total, 100), evidence

def find_mother(infant, mothers_list, threshold=50, top_n=3):
    phone_usage_map = _build_phone_usage_map(mothers_list)
    candidates = []
    for mother in mothers_list:
        score, evidence = score_match(infant, mother, phone_usage_map)
        if score >= threshold:
            art_id = ""
            for ident in mother.get("identifier",[]):
                if "art" in ident.get("system","").lower(): art_id = ident["value"]
            candidates.append(LinkageResult(
                mother_id=mother["id"], mother_name=f"{mother['name'][0]['given'][0]} {mother['name'][0]['family']}",
                art_id=art_id, confidence=score/100, evidence=evidence))
    if not candidates: return []
    candidates.sort(key=lambda x: x.confidence, reverse=True)
    return candidates[:top_n]

print('✅ Forensic Linker ready')

## 5. Agent 2 — Adherence Miner (Gemma 4 + Hallucination Firewall)
Uses Gemma 4 **native function calling** via Ollama to extract adherence risks from clinical notes. Every output is verified by the **Hallucination Firewall** — if the `source_quote` doesn't exist in the original text, it's rejected.

In [ ]:
import json, time, requests
from rapidfuzz import fuzz as adherence_fuzz

OLLAMA_BASE_URL = 'http://localhost:11434'
GEMMA_MODEL = 'gemma4:e4b'
GEMMA_READ_TIMEOUT = 180
GEMMA_MAX_RETRIES = 2

SYSTEM_PROMPT = """You are a clinical adherence analyst for HIV care in Sub-Saharan Africa.
Given a single clinical note, extract any adherence risk indicators.
Look for: missed pharmacy pick-ups, missed ANC visits, transport or financial difficulties,
self-reported missed ART doses, late presentation to care, and any other adherence barrier.

You MUST respond with a JSON object in this exact format:
{"risks": [{"indicator": "short label", "severity": "low|moderate|high", "source_quote": "exact text from note"}]}

If no risks are found, respond with: {"risks": []}
IMPORTANT: source_quote MUST be copied directly from the note."""

def _check_ollama():
    try:
        resp = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3)
        if resp.status_code == 200:
            model_names = [model.get('name', '') for model in resp.json().get('models', [])]
            return GEMMA_MODEL in model_names
    except Exception:
        pass
    return False

HAS_GEMMA = _check_ollama()
print(f'Gemma 4 available ({GEMMA_MODEL}): {HAS_GEMMA}')

def _hallucination_firewall(risks_data, note_text):
    """Verify every source_quote is grounded in the original note (fuzzy match >= 70%)."""
    verified = []
    for risk in risks_data:
        quote = risk.get('source_quote', '')
        if not quote:
            print(f"  🚫 FIREWALL REJECTED: empty source_quote")
            continue
        similarity = adherence_fuzz.partial_ratio(quote.lower(), note_text.lower())
        if similarity >= 70:
            verified.append(risk)
        else:
            print(f"  🚫 FIREWALL REJECTED ({similarity}% match): \"{quote[:60]}\"")
    return verified

def _dedupe_risks(risks):
    seen = set()
    unique = []
    for risk in risks:
        key = (risk.indicator, risk.source_text, risk.source_date, risk.severity)
        if key not in seen:
            seen.add(key)
            unique.append(risk)
    return unique

def _to_adherence_risks(risks_data, note):
    """Convert raw risk dicts to AdherenceRisk objects using the specific note's date."""
    risks = []
    for risk in risks_data:
        risks.append(
            AdherenceRisk(
                indicator=risk['indicator'],
                source_text=risk.get('source_quote', ''),
                source_date=note.get('date', 'unknown'),
                severity=risk.get('severity', 'moderate'),
            )
        )
    return risks

def _call_gemma_for_note(note_text):
    """Send a single clinical note to Gemma using JSON output mode and return raw risk dicts."""
    payload = {
        'model': GEMMA_MODEL,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Analyze this clinical note and return JSON:\n\n{note_text}'},
        ],
        'format': 'json',
        'stream': False,
        'keep_alive': '30m',
        'options': {'temperature': 0.1, 'num_predict': 512},
    }

    last_error = None
    for attempt in range(1, GEMMA_MAX_RETRIES + 1):
        try:
            resp = requests.post(
                f'{OLLAMA_BASE_URL}/api/chat',
                json=payload,
                timeout=(5, GEMMA_READ_TIMEOUT),
            )
            resp.raise_for_status()
            result = resp.json()
            content = result.get('message', {}).get('content', '').strip()
            if not content:
                return []

            parsed = json.loads(content)

            # Handle {"risks": [...]} wrapper
            if isinstance(parsed, dict) and 'risks' in parsed:
                risks = parsed['risks']
                if isinstance(risks, list):
                    return risks
                return []

            # Handle bare array [...]
            if isinstance(parsed, list):
                return parsed

            return []

        except Exception as error:
            last_error = error
            print(f'  ⚠️ Gemma attempt {attempt}/{GEMMA_MAX_RETRIES} failed: {error}')
            if attempt < GEMMA_MAX_RETRIES:
                time.sleep(2)

    raise RuntimeError(f'Gemma failed after {GEMMA_MAX_RETRIES} retries: {last_error}')

# --- Counters for tracking Gemma vs fallback contribution ---
_gemma_found = 0
_fallback_found = 0

def extract_adherence_risks(clinical_notes):
    """Extract adherence risks by sending each note to Gemma individually."""
    global _gemma_found, _fallback_found
    if not clinical_notes:
        return []

    all_risks = []
    for note in clinical_notes:
        note_text = note.get('content', '')
        if not note_text:
            continue

        # Try Gemma first for this individual note
        try:
            raw_risks = _call_gemma_for_note(note_text)
            verified = _hallucination_firewall(raw_risks, note_text)
            if verified:
                note_risks = _to_adherence_risks(verified, note)
                all_risks.extend(note_risks)
                _gemma_found += len(note_risks)
                continue
        except Exception as e:
            print(f'  ⚠️ Gemma error for note: {e}')

        # Per-note keyword fallback only if Gemma produced nothing usable
        fallback = extract_adherence_risks_offline([note])
        if fallback:
            all_risks.extend(fallback)
            _fallback_found += len(fallback)

    return _dedupe_risks(all_risks)

def extract_adherence_risks_offline(clinical_notes):
    risks = []
    keywords = {
        'missed pharmacy': ('Missed pharmacy pick-up', 'moderate'),
        'missed pick-up': ('Missed pharmacy pick-up', 'moderate'),
        'transport': ('Transport difficulties reported', 'moderate'),
        'financial': ('Financial barriers to care', 'moderate'),
        'missing art': ('Missed ART doses', 'high'),
        'skipping medication': ('Self-reported medication skipping', 'high'),
        'missed doses': ('Missed ART doses', 'high'),
        'missing doses': ('Missed ART doses', 'high'),
        'late presentation': ('Late entry to antenatal care', 'moderate'),
        'did not attend': ('Missed scheduled visit', 'moderate'),
        'first visit at 3': ('Late entry to antenatal care', 'moderate'),
    }
    for note in clinical_notes:
        content = note.get('content', '').lower()
        for keyword, (indicator, severity) in keywords.items():
            if keyword in content:
                risks.append(
                    AdherenceRisk(
                        indicator=indicator,
                        source_text=note['content'],
                        source_date=note.get('date', 'unknown'),
                        severity=severity,
                    )
                )
    return _dedupe_risks(risks)

print('✅ Adherence Miner ready (per-note Gemma JSON mode + fuzzy firewall)')


## 6. Agent 3 — Protocol Guardian
Deterministic risk classification with **MODERATE tier**, **Missing VL → HIGH (Rule 0)**, and **severity-weighted scoring**.

In [ ]:
def calculate_adherence_score(adherence_risks):
    weights = {"high": 3, "moderate": 2, "low": 1}
    return sum(weights.get(r.severity, 2) for r in adherence_risks)

def classify_risk(mother, adherence_risks):
    vl_data = mother.get("viral_load", {})
    vl_value = vl_data.get("valueQuantity", {}).get("value", None)
    vl_date = vl_data.get("effectiveDateTime", "")
    reasons = []
    # Rule 0: No VL data
    if vl_value is None or (vl_value == 0 and not vl_date):
        reasons.append("No viral load on record — treat as untreated/unknown status")
        return RiskClassification(level="HIGH", reasons=reasons, viral_load=0, viral_load_date="", adherence_risks=adherence_risks)
    if vl_value > 1000:
        reasons.append(f"Unsuppressed viral load: {vl_value} copies/mL")
    if vl_date:
        try:
            days_old = (datetime.now() - datetime.fromisoformat(vl_date)).days
            if days_old > 90: reasons.append(f"Last VL test is {days_old} days old (>3 months — stale)")
        except ValueError: reasons.append("Unable to parse VL date — treating as stale")
    else: reasons.append("No VL test date available — treating as stale")
    adherence_score = calculate_adherence_score(adherence_risks)
    if vl_value <= 1000 and adherence_score >= 4:
        reasons.append(f"Adherence score {adherence_score} (≥4) despite suppressed VL")
    if reasons: level = "HIGH"
    elif 50 <= vl_value <= 1000:
        level = "MODERATE"; reasons.append(f"Borderline viral load: {vl_value} copies/mL (50–1000)")
    elif len(adherence_risks) >= 1:
        level = "MODERATE"; reasons.append(f"{len(adherence_risks)} adherence risk(s) (score: {adherence_score})")
    else: level = "LOW"
    return RiskClassification(level=level, reasons=reasons, viral_load=vl_value, viral_load_date=vl_date, adherence_risks=adherence_risks)

def get_prophylaxis_recommendation(risk_level):
    if risk_level == "HIGH":
        return {"regimen": "Triple Therapy: AZT + 3TC + NVP (or Raltegravir)", "duration": "6 weeks",
                "urgency": "Start within 6 hours of birth", "follow_up": "Continue AZT to complete 6 weeks"}
    elif risk_level == "MODERATE":
        return {"regimen": "Assess: Consider Triple Therapy or AZT alone", "duration": "2–6 weeks",
                "urgency": "Start within 6 hours — clinician to determine", "follow_up": "Obtain current VL; reassess"}
    else:
        return {"regimen": "AZT (Zidovudine) only", "duration": "2 weeks",
                "urgency": "Start within 6 hours of birth", "follow_up": "Standard follow-up"}

def build_bridge_summary(infant, linkage, risk, audit_hash=''):
    infant_name = f"{infant['name'][0].get('given',[''])[0]} {infant['name'][0].get('family','')}".strip()
    evidence_summary = "; ".join(e.detail for e in linkage.evidence)
    adherence_findings = [f"{r.indicator} [{r.severity.upper()}] ({r.source_date})" for r in risk.adherence_risks]
    prophylaxis = get_prophylaxis_recommendation(risk.level)
    if risk.level == "HIGH":
        action = f"URGENT: {prophylaxis['regimen']} for {prophylaxis['duration']}. {prophylaxis['urgency']}."
    elif risk.level == "MODERATE":
        action = f"REVIEW: {prophylaxis['regimen']}. {prophylaxis['urgency']}."
    else:
        action = f"Standard: {prophylaxis['regimen']} for {prophylaxis['duration']}. {prophylaxis['urgency']}."
    return BridgeSummary(infant_name=infant_name, mother_name=linkage.mother_name, art_id=linkage.art_id,
                         confidence=linkage.confidence, evidence_summary=evidence_summary, viral_load=risk.viral_load,
                         risk_level=risk.level, adherence_findings=adherence_findings, recommended_action=action,
                         audit_hash=audit_hash)

print('✅ Protocol Guardian ready')

## 7. Security — Role-Based Access & Hash-Chained Audit Log
NDPA 2023 compliant: nurses see only risk flags, specialists see full details. Every action is logged with SHA-256 hash chaining.

In [ ]:
import hashlib, os
from datetime import UTC

# --- Role-Based Access ---
@dataclass
class AuthContext:
    user_id: str
    role: str
    facility_id: str
    organization: str
    token: str = ""

ROLE_ACCESS = {
    "hiv_specialist": {"can_view_notes": True, "can_view_mother_identity": True, "can_view_viral_load": True,
                       "can_confirm_linkage": True, "can_view_adherence_details": True},
    "nurse": {"can_view_notes": False, "can_view_mother_identity": False, "can_view_viral_load": False,
              "can_confirm_linkage": False, "can_view_adherence_details": False},
    "facility_manager": {"can_view_notes": False, "can_view_mother_identity": True, "can_view_viral_load": False,
                         "can_confirm_linkage": False, "can_view_adherence_details": False},
}

def filter_output_by_role(bridge_summary, auth):
    perms = ROLE_ACCESS.get(auth.role, {})
    if not perms: return {"error": "ACCESS DENIED", "role": auth.role}
    output = {"risk_level": bridge_summary.risk_level, "recommended_action": bridge_summary.recommended_action,
              "infant_name": bridge_summary.infant_name}
    if auth.role == "nurse": return output
    if auth.role == "facility_manager":
        output["mother_name"] = bridge_summary.mother_name; output["confidence"] = bridge_summary.confidence; return output
    if auth.role == "hiv_specialist":
        output.update({"mother_name": bridge_summary.mother_name, "art_id": bridge_summary.art_id,
                       "confidence": bridge_summary.confidence, "evidence_summary": bridge_summary.evidence_summary,
                       "viral_load": bridge_summary.viral_load, "adherence_findings": bridge_summary.adherence_findings})
    return output

# --- Hash-Chained Audit Log (NDPA 2023 Compliant) ---
AUDIT_LOG = []  # In-memory for Kaggle notebook
GENESIS_HASH = "0" * 64
_last_hash = GENESIS_HASH

def _compute_hash(entry_json, prev_hash):
    return hashlib.sha256(f"{prev_hash}|{entry_json}".encode("utf-8")).hexdigest()

def audit_log(event, **kwargs):
    global _last_hash
    entry = {"event": event, "timestamp": datetime.now(UTC).isoformat().replace('+00:00', 'Z'), **kwargs}
    entry["prev_hash"] = _last_hash
    entry_json = json.dumps(entry, sort_keys=True, default=str)
    entry["hash"] = _compute_hash(entry_json, _last_hash)
    _last_hash = entry["hash"]
    AUDIT_LOG.append(entry)
    return entry["hash"]

def get_last_hash():
    return _last_hash

def verify_audit_chain():
    for i, entry in enumerate(AUDIT_LOG):
        stored_hash = entry.get("hash","")
        prev_hash = entry.get("prev_hash", GENESIS_HASH)
        check = {k:v for k,v in entry.items() if k != "hash"}
        expected = _compute_hash(json.dumps(check, sort_keys=True, default=str), prev_hash)
        if stored_hash != expected: return {"valid": False, "broken_at": i}
    return {"valid": True, "entries": len(AUDIT_LOG)}

print('✅ Security layer ready')


## 8. 🚀 End-to-End Pipeline Demo
Process all 50 newborns through the full VIGILANT pipeline.

In [ ]:
results = {"HIGH": [], "MODERATE": [], "LOW": [], "UNLINKED": []}
_gemma_found = 0
_fallback_found = 0

for infant in newborns:
    # Step 1: Forensic Linker — find mother
    candidates = find_mother(infant, mothers)
    if not candidates:
        results["UNLINKED"].append(infant)
        continue
    best = candidates[0]
    
    # Step 2: Get mother record
    mother = next((m for m in mothers if m["id"] == best.mother_id), None)
    if not mother:
        results["UNLINKED"].append(infant)
        continue
    
    # Step 3: Adherence Miner — extract risks from clinical notes
    adherence_risks = extract_adherence_risks(mother.get("clinical_notes", []))
    
    # Step 4: Protocol Guardian — classify risk
    risk = classify_risk(mother, adherence_risks)
    
    # Step 5: Audit log
    audit_log("linkage", infant_id=infant["id"], mother_id=best.mother_id, confidence=best.confidence)
    audit_hash = audit_log("risk_classification", infant_id=infant["id"], risk_level=risk.level)
    
    # Step 6: Build Bridge Summary with tamper-proof audit hash
    summary = build_bridge_summary(infant, best, risk, audit_hash=get_last_hash())
    
    results[risk.level].append(summary)

adherence_summary_count = sum(
    1 for bucket in ["HIGH", "MODERATE", "LOW"] for summary in results[bucket] if summary.adherence_findings
)

print("="*60)
print("🛡️ VIGILANT Pipeline Results")
print("="*60)
print(f"  🔴 HIGH risk:     {len(results['HIGH'])}")
print(f"  🟡 MODERATE risk: {len(results['MODERATE'])}")
print(f"  🟢 LOW risk:      {len(results['LOW'])}")
print(f"  📝 With adherence findings: {adherence_summary_count}")
print(f"  🤖 Gemma extracted:   {_gemma_found} risks")
print(f"  🔧 Keyword fallback:  {_fallback_found} risks")
print(f"  ❓ Unlinked:      {len(results['UNLINKED'])}")
print(f"  📋 Audit entries: {len(AUDIT_LOG)}")
print(f"  🔗 Chain valid:   {verify_audit_chain()}")


## 9. Detailed Case Examples

In [ ]:
# Show a HIGH risk case driven by viral load alone
if results["HIGH"]:
    case = results["HIGH"][0]
    print("🔴 HIGH RISK CASE (VL-driven)")
    print(f"  Infant: {case.infant_name}")
    print(f"  Mother: {case.mother_name} (ART: {case.art_id})")
    print(f"  Confidence: {case.confidence:.0%}")
    print(f"  Evidence: {case.evidence_summary}")
    print(f"  Viral Load: {case.viral_load}")
    print(f"  Adherence: {case.adherence_findings}")
    print(f"  ⚡ ACTION: {case.recommended_action}")
    if case.audit_hash:
        print(f"  🛡️ Tamper-Proof Audit ID: {case.audit_hash[:8]}...{case.audit_hash[-4:]}")

print()

# Show a HIGH or MODERATE case with extracted adherence findings, if present
adherence_case = None
for bucket in ["HIGH", "MODERATE"]:
    adherence_case = next((summary for summary in results[bucket] if summary.adherence_findings), None)
    if adherence_case:
        break

if adherence_case:
    print("🟠 CASE WITH ADHERENCE FINDINGS")
    print(f"  Infant: {adherence_case.infant_name}")
    print(f"  Mother: {adherence_case.mother_name} (ART: {adherence_case.art_id})")
    print(f"  Confidence: {adherence_case.confidence:.0%}")
    print(f"  Viral Load: {adherence_case.viral_load}")
    print(f"  Adherence: {adherence_case.adherence_findings}")
    print(f"  ⚡ ACTION: {adherence_case.recommended_action}")
else:
    print("⚠️ No cases with adherence findings were found in this run.")

print()

# Show role-based filtering
if results["HIGH"]:
    case = results["HIGH"][0]
    print("👩‍⚕️ NURSE VIEW (data minimization):")
    nurse = AuthContext(user_id="nurse-001", role="nurse", facility_id="F1", organization="APIN")
    print(json.dumps(filter_output_by_role(case, nurse), indent=2))
    print()
    print("👨‍⚕️ HIV SPECIALIST VIEW (full access):")
    specialist = AuthContext(user_id="dr-001", role="hiv_specialist", facility_id="F1", organization="APIN")
    print(json.dumps(filter_output_by_role(case, specialist), indent=2))

## 10. Validation Tests

In [ ]:
# Test 1: Missing VL → HIGH (Rule 0)
missing_vl_mom = {"viral_load": {"valueQuantity": {}, "effectiveDateTime": ""}}
r = classify_risk(missing_vl_mom, [])
assert r.level == "HIGH", f"Expected HIGH, got {r.level}"
print("✅ Test 1: Missing VL → HIGH")

# Test 2: MODERATE tier (VL 50-1000)
mod_mom = {"viral_load": {"valueQuantity": {"value": 500}, "effectiveDateTime": datetime.now().isoformat()}}
r = classify_risk(mod_mom, [])
assert r.level == "MODERATE", f"Expected MODERATE, got {r.level}"
print("✅ Test 2: Borderline VL → MODERATE")

# Test 3: Severity-weighted scoring (score ≥ 4 → HIGH)
risks = [AdherenceRisk("r1","","","high"), AdherenceRisk("r2","","","moderate")]
assert calculate_adherence_score(risks) == 5
print("✅ Test 3: Severity scoring (3+2=5)")

# Test 4: Hallucination firewall
fake = [{"source_quote": "this text does not exist", "indicator": "fake", "severity": "high"}]
verified = _hallucination_firewall(fake, "actual clinical note text")
assert len(verified) == 0
print("✅ Test 4: Hallucination firewall rejects fabricated quotes")

# Test 5: Audit chain integrity
chain_result = verify_audit_chain()
assert chain_result["valid"] == True
print(f"✅ Test 5: Audit chain valid ({chain_result['entries']} entries)")

# Test 6: Shared phone penalty
phone_map = {"+234-800-SHARED": 5}
penalty_weight = max(5, 15 // 5)
assert penalty_weight < 15
print(f"✅ Test 6: Shared phone penalty (15 → {penalty_weight})")

print("\n🎉 All validation tests passed!")

## 11. Architecture Summary

```
┌─────────────────────────────────────────────────────────┐
│                    VIGILANT System                       │
│                  (Gemma 4 via Ollama)                    │
├─────────────┬──────────────────┬────────────────────────┤
│  Forensic   │   Adherence      │    Protocol            │
│  Linker     │   Miner          │    Guardian            │
│             │                  │                        │
│ • Fuzzy     │ • Gemma 4        │ • Missing VL→HIGH     │
│   name      │   function       │ • MODERATE tier        │
│   matching  │   calling        │ • Severity-weighted    │
│ • Top-3     │ • Hallucination  │   scoring              │
│   candidates│   firewall       │ • Drug regimens        │
│ • Shared    │ • Offline        │   (AZT+3TC+NVP)       │
│   phone     │   fallback       │ • 6-hour urgency       │
│   penalty   │                  │                        │
├─────────────┴──────────────────┴────────────────────────┤
│  Security: Role-Based Access + SHA-256 Audit Chain      │
│  Privacy: NDPA 2023 / POPIA / Kenya DPA compliant       │
│  All processing LOCAL — no data leaves the device       │
└─────────────────────────────────────────────────────────┘
```